# Langgraph – wprowadzenie

Langgraph to biblioteka pozwalająca wykonywać pipeline'y składające się z wielu etapów, tak zwanych node'ów. Zwykle używamy go do zarządzania flow procesów opartych o integrację z modelami LLM ale nie jest to konieczne

## Pipeline state

Pipeline state to klasa dziedzicząca po `TypedDict` który odpowiada stanowi flow, który jest updatowany z każdym nodem pipeline'u.

W poniższym przykładzie będziemy wczytywać i przetwarzać dane z pliku tekstowego.

In [ ]:
from typing import TypedDict


class State(TypedDict):
    file_path: str
    raw_content: str
    parsed_entries: list[dict[str, str]]
    filtered_entries: list[dict[str, str]]
    metrics: dict[str, int]

## Node'y

Pipeline składa się z node'ów, które są połączone krawędziami. W node'ach odbywa się updatowanie stanu pipeline'u.

In [ ]:
from langchain_core.runnables import RunnableConfig

In [ ]:
debug = True

In [ ]:
def read_log_file_node(state: State):
    if debug:
        print("Node: read_log_file |", "State:", state, "\n")
    
    file_path = state["file_path"]
    
    with open(file_path, "r", encoding="utf-8") as f:
        content = f.read()
    return {"raw_content": content}

In [ ]:
def parse_log_entries_node(state: State):
    if debug:
        print("Node: parse_log_entries |", "State:", state, "\n")

    lines = state["raw_content"].splitlines()
    
    entries = []
    for line in lines:
        if not line.strip():
            continue
        parts = line.split(" | ")
        if len(parts) == 4:
            entries.append({
                "timestamp": parts[0].strip(),
                "level": parts[1].strip(),
                "ip": parts[2].strip(),
                "message": parts[3].strip()
            })
    return {"parsed_entries": entries}

In [ ]:
def filter_security_alerts_node(state: State, config: RunnableConfig):
    if debug:
        print("Node: filter_security_alerts |", "State:", state, "| Config:", config, "\n")

    configurable = config.get("configurable", {})
    target_level = configurable.get("alert_level", "CRITICAL")

    filtered = [
        entry for entry in state["parsed_entries"]
        if entry["level"] == target_level
    ]
    return {"filtered_entries": filtered}

In [ ]:
def calculate_metrics_node(state: State):
    if debug:
        print("Node: calculate_metrics |", "State:", state, "\n")
    
    entries = state["filtered_entries"]
    
    ip_counts = {}
    for entry in entries:
        ip = entry["ip"]
        ip_counts[ip] = ip_counts.get(ip, 0) + 1
    return {"metrics": ip_counts}

## State graph

In [ ]:
from langchain_core.runnables import RunnableConfig
from langgraph.graph import StateGraph, START, END

In [ ]:
graph = StateGraph(State)

graph.add_node("read_file", read_log_file_node)
graph.add_node("parse_log_entries", parse_log_entries_node)
graph.add_node("filter_security_alerts", filter_security_alerts_node)
graph.add_node("calculate_metrics", calculate_metrics_node)

graph.add_edge(START, "read_file")
graph.add_edge("read_file", "parse_log_entries")
graph.add_edge("parse_log_entries", "filter_security_alerts")
graph.add_edge("filter_security_alerts", "calculate_metrics")
graph.add_edge("calculate_metrics", END)

pipeline = graph.compile()

In [ ]:
from IPython.display import Image, display

display(Image(pipeline.get_graph().draw_mermaid_png(), width=120))

## Uruchomienie pipeline'u

In [ ]:
initial_state = {"file_path": "log_file.log"}
runtime_config = {"configurable": {"alert_level": "CRITICAL"}}

result = pipeline.invoke(initial_state, config=runtime_config)

In [ ]:
result["metrics"]

In [ ]:
result.keys()